# The core: querying `p(any column | any subset)` directly

Every estimator in `sklm` is a thin adapter over one object, `TabularLanguageModel`: a single
fine-tune over randomly permuted column orders (introduced in `00-intro.ipynb`) that learns

$$ p\!\left(x_j \mid x_S\right), \qquad j \notin S,\; S \subseteq \{1,\dots,d\}. $$

The classifier, regressor, imputer and oversampler are just different choices of which columns go
in $S$ (the prompt) and which is $x_j$ (the output). This notebook drives the core object directly
and performs both operations the estimators rely on:

- **Score** a fixed candidate set for one column — what the classifier does — with
  `predict_proba(known, target, candidates)`, returning a normalized distribution
  (softmax over per-candidate likelihoods, as in `01-iris-classifier.ipynb`).
- **Generate** a value for another column — what the imputer and regressor do — with
  `complete(known, targets, generation)`.

Because the *same* model answers both kinds of query after one fit, you can ask questions no
single supervised model was trained for: `p(species | small petals)` and a sampled
`petal length | species=setosa` come from the same parameters.

In [1]:
from sklearn.datasets import load_iris

from sklm import (
    GenerationConfig,
    JSONSerializer,
    JupyterCallback,
    MLXBackend,
    ModelConfig,
    TabularLanguageModel,
    TrainingConfig,
)

SEED = 42

## Data

The full Iris table — four measurements plus the species label — with no target singled out.

In [2]:
iris = load_iris(as_frame=True)
frame = iris.data.round(1)
frame["species"] = iris.target_names[iris.target]
frame.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


## Fit the core model

`TabularLanguageModel.fit` with no `target_cols` treats *every* column as a potential target — the
permutation augmentation does the rest.

In [3]:
lm = TabularLanguageModel(
    backend=MLXBackend(),
    serializer=JSONSerializer(),
    model=ModelConfig(model="gabfssilva/distilgpt2"),
    training=TrainingConfig(epochs=40, batch_size=16),
    callback=JupyterCallback(),
    random_state=SEED,
).fit(frame)

## Query 1 — score a categorical column (classification)

Condition only on the two petal measurements and ask for the species distribution. This is exactly
what `LanguageModelClassifier` does internally.

In [4]:
known = {"petal length (cm)": 1.4, "petal width (cm)": 0.2}
proba = lm.predict_proba(known, "species", list(iris.target_names))
for c, p in zip(iris.target_names, proba, strict=True):
    print(f"p(species={c} | small petals) = {p:.3f}")

p(species=setosa | small petals) = 0.810
p(species=versicolor | small petals) = 0.154
p(species=virginica | small petals) = 0.036


## Query 2 — generate a numeric column (imputation / regression)

Now flip it around: condition on the species and *generate* a petal length. `complete` returns a
dict of the generated columns, or `None` if the output stayed malformed after the retries.

In [5]:
gen = GenerationConfig(max_new_tokens=8)
out = lm.complete({"species": "setosa"}, ["petal length (cm)"], gen)
sampled = out["petal length (cm)"] if out is not None else "(malformed)"
print(f"sampled petal length | species=setosa -> {sampled}")

sampled petal length | species=setosa -> 1.5
